# 01 · Define & Explore — enzyme design, the Kemp reaction, and the theozyme

**Standard slot:** *define & explore.* **For Project 18 this means:** understand de novo enzyme
design and the Kemp elimination, then **construct the theozyme** (the catalytic functional groups +
transition-state geometry) and run a mock theozyme→scaffold hello-world (D0).

Run `00_setup.ipynb` first in this session.

## Why the Kemp elimination is *the* de novo enzyme benchmark
The Kemp elimination is the base-catalysed ring opening of a benzisoxazole. Two properties make it
the field's model reaction:
- **No natural counterpart** — there is no evolved Kemp eliminase, so any activity you design is
  genuinely *de novo* (not borrowed from a natural template).
- **Simple UV readout** — the substrate **5-nitrobenzisoxazole** is chromogenic; the product
  absorbs, so kinetics are a one-step plate-reader assay.

The honest history: the first computational Kemp eliminases (Röthlisberger 2008) were weak and only
became efficient after **directed evolution**. Recent methods (Riff-Diff, RFdiffusion2) reach
near-natural rates *without* evolution — which is exactly the methods claim this project tests.

## The theozyme — the catalytic motif you must build
A **theozyme** ("theoretical enzyme") is the minimal set of catalytic functional groups placed
around the **transition state**. For the Kemp elimination the canonical motif is:

| Role | Residue(s) | Job in the TS |
|------|-----------|----------------|
| catalytic base | Asp / Glu | abstracts the C3 proton |
| π-stack | Trp / Tyr / Phe | binds & orients the planar substrate; delocalises developing charge |
| H-bond donor | Ser / Thr / backbone amide | stabilises the developing phenolate / nitro oxygen |

You **construct** this from the literature and/or a QM transition-state model — it is a teaching
template (`data/inputs/theozyme_def.txt`), **not** fabricated experimental data. Place groups around
the **transition state**, not the ground state.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the theozyme (mock hello-world)
`scripts/enzyme_tools.py` exposes `build_theozyme(reaction)` → a functional-group geometry spec.
The distances/angles it ships are **PLACEHOLDERS** — replace them in `data/inputs/theozyme_def.txt`
(and in `build_theozyme`) with real, cited values during P1. This is the **enzyme-family template**
hook: Projects 19/21 (Ser-His-Asp triad), 20 (Zn-His3-OH), 24 (cofactor) override the motif here.

In [ ]:
from enzyme_tools import build_theozyme

theo = build_theozyme("kemp_elimination")
print("Reaction :", theo.reaction)
print("Substrate:", theo.substrate)
print("Provenance:", theo.provenance)
print("\nCatalytic functional groups (PLACEHOLDER geometry — fill from literature/QM):")
for fg in theo.functional_groups:
    ang = f"{fg.target_angle}deg" if fg.target_angle is not None else "n/a"
    print(f"  {fg.role:14s} {fg.residue}/{fg.atom:4s}  d={fg.target_distance}A  angle={ang}")
print("\nCatalytic residues to FIX during sequence design:", theo.catalytic_residue_ids())

## A first mock scaffold + sequence (no GPU)
`scaffold_motif(...)` (mock) returns placeholder backbones presenting the motif;
`ligandmpnn_fix_catalytic(...)` (mock) designs sequences with the catalytic residues fixed. **Every
number here is SYNTHETIC** — this only proves the plumbing runs anywhere. Switch to the real
backends (RFdiffusion2/Riff-Diff on an A100; LigandMPNN CPU-fast) in `02_generate.ipynb`.

In [ ]:
from enzyme_tools import scaffold_motif, ligandmpnn_fix_catalytic, catalytic_geometry_rmsd

scaffolds = scaffold_motif(theo, n=5, method="mock")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_fix_catalytic(scaffolds[0], theo.catalytic_residue_ids(), n=3, tool="mock")
print(f"\n{len(seqs)} mock sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_catalytic_roles']})")

cg = catalytic_geometry_rmsd(None, theo)   # mock, SYNTHETIC
print(f"\ncatalytic_geometry_rmsd (mock, SYNTHETIC) = {cg} A  -> pass if < 0.5 A")
print("NOTE: these are placeholder numbers. The real campaign is in notebook 02.")

## The metrics that decide an enzyme design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | activity |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / catalysis |
| pLDDT (catalytic) | ≥ 90 | confidence *at the active site* | the geometry is correct |
| **catalytic_geom_rmsd** | **< 0.5 Å** | predicted catalytic atoms vs the theozyme | **activity** (the design can still be dead) |

The fourth row is the point of the whole project — and the last column is the message to never
forget: **in-silico catalytic geometry does not guarantee a working enzyme.** Only a kinetic assay
does (notebook 05).

## D0 checklist
- [ ] Half-page on de novo enzyme design + the honest Kemp hit-rate history.
- [ ] 1-page problem statement with **measurable** success criteria + the controls you'll need.
- [ ] Theozyme spec started in `data/inputs/theozyme_def.txt` (replace the PLACEHOLDERs, cite sources).
- [ ] Reproduced mock hello-world (functional-group spec + a mock scaffold record).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the motif and run LigandMPNN with the catalytic residues fixed.